<a href="https://colab.research.google.com/github/AliefReefandi/Grokking-Deep-Learning/blob/main/Chapter_13_Introducing_Automatic_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13 — Introducing Automatic Optimization

## Tujuan Pembelajaran

Pada chapter ini kita mulai berpindah dari implementasi neural network secara manual menuju penggunaan **deep learning framework**.

Framework deep learning seperti PyTorch dan TensorFlow dibuat untuk menyederhanakan proses:
- Forward propagation
- Backpropagation
- Perhitungan gradient
- Optimasi parameter
- Penyusunan neural network menggunakan berbagai layer

Di chapter ini kita tidak langsung menggunakan PyTorch. Kita justru membangun framework sederhana sendiri agar memahami apa yang sebenarnya terjadi "di balik layar".

Framework membantu mengurangi kode yang berulang, mengurangi kesalahan, mempercepat pengembangan, dan dapat meningkatkan performa ketika menggunakan GPU.

Konsep utama chapter ini adalah:

**Tensor → Autograd → Backpropagation → Optimizer → Layer → Neural Network**

Sumber: *Grokking Deep Learning*, Chapter 13.

## 1. Tensor

Tensor adalah struktur data yang digunakan untuk menyimpan angka dalam berbagai dimensi.

Secara sederhana:

- Scalar → 1 angka
- Vector → 1 dimensi
- Matrix → 2 dimensi
- Tensor → dapat memiliki lebih dari 2 dimensi

Dalam deep learning, hampir semua data direpresentasikan sebagai tensor.

Contoh:

x = [1, 2, 3]

merupakan vector.

Sedangkan:

x = [[1, 2, 3],
     [4, 5, 6]]

merupakan matrix.

Framework deep learning menggunakan tensor sebagai objek utama untuk melakukan operasi matematika dan menyimpan gradient.

In [1]:
import numpy as np

# Scalar
scalar = np.array(5)

# Vector
vector = np.array([1, 2, 3])

# Matrix
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print("Scalar:")
print(scalar)

print("\nVector:")
print(vector)

print("\nMatrix:")
print(matrix)

print("\nShape matrix:", matrix.shape)

Scalar:
5

Vector:
[1 2 3]

Matrix:
[[1 2 3]
 [4 5 6]]

Shape matrix: (2, 3)


## 2. Automatic Gradient Computation (Autograd)

Pada neural network, kita perlu menghitung gradient untuk mengetahui bagaimana parameter harus diperbarui.

Sebelumnya kita menghitung gradient secara manual.

Dengan autograd, framework dapat:

1. Menyimpan operasi yang dilakukan pada tensor.
2. Membentuk hubungan antar operasi.
3. Menelusuri operasi tersebut secara backward.
4. Menghitung gradient secara otomatis.

Contoh sederhana:

a → perkalian → b → penjumlahan → c

Jika kita ingin mengetahui bagaimana c berubah terhadap a, autograd akan menelusuri kembali operasi tersebut menggunakan aturan chain rule.

Inilah dasar dari automatic backpropagation.

In [2]:
# Contoh sederhana konsep autograd

x = 3.0
y = 2.0

# Forward
z = x * y

print("x =", x)
print("y =", y)
print("z =", z)

# Secara matematis:
# z = x * y
# dz/dx = y
# dz/dy = x

dz_dx = y
dz_dy = x

print("dz/dx =", dz_dx)
print("dz/dy =", dz_dy)

x = 3.0
y = 2.0
z = 6.0
dz/dx = 2.0
dz/dy = 3.0


## 3. Tensor yang Digunakan Berkali-kali

Salah satu masalah penting dalam autograd adalah ketika satu tensor digunakan dalam beberapa operasi.

Contoh:

a → operasi 1
  → operasi 2

Gradient dari kedua jalur tersebut harus dikumpulkan.

Karena itu, sistem autograd harus mengetahui:
- tensor mana yang menghasilkan tensor baru,
- operasi apa yang dilakukan,
- dan bagaimana gradient diteruskan kembali.

Dalam framework sederhana pada buku, informasi ini disimpan melalui konsep seperti:

- creators
- creation_op
- grad

Dengan demikian, setiap Tensor mengetahui dari operasi mana dirinya berasal.

## 4. Backpropagation pada Operasi Penjumlahan

Misalkan:

c = a + b

Jika kita mengetahui gradient dari c, maka gradient tersebut diteruskan ke a dan b.

Karena:

∂c/∂a = 1
∂c/∂b = 1

maka:

gradient_a = gradient_c
gradient_b = gradient_c

Ini terlihat sederhana, tetapi konsep yang sama digunakan sebagai bagian dari backpropagation pada neural network yang jauh lebih kompleks.

In [3]:
# Simulasi backpropagation pada operasi penjumlahan

a = 5
b = 3

c = a + b

# Misalkan gradient dari c adalah 1
grad_c = 1

grad_a = grad_c
grad_b = grad_c

print("c =", c)
print("gradient a =", grad_a)
print("gradient b =", grad_b)

c = 8
gradient a = 1
gradient b = 1


## 5. Operasi Negasi

Jika:

y = -x

maka:

dy/dx = -1

Jadi gradient yang diteruskan ke x akan berubah tanda.

Contoh:

x = 5
y = -5

Jika gradient y = 1, maka:

gradient_x = -1

Konsep sederhana ini digunakan untuk membangun operasi yang lebih kompleks pada autograd.

In [4]:
x = 5
y = -x

grad_y = 1
grad_x = -grad_y

print("x =", x)
print("y =", y)
print("gradient x =", grad_x)

x = 5
y = -5
gradient x = -1


## 6. Menambahkan Operasi Matematika

Framework yang dibangun pada chapter ini kemudian diperluas agar dapat menangani berbagai operasi:

- Addition (+)
- Subtraction (-)
- Multiplication (*)
- Sum
- Expand
- Transpose
- Matrix multiplication (mm)

Setiap operasi membutuhkan dua hal:

1. Forward propagation
2. Backward propagation

Contohnya untuk perkalian:

y = a × b

gradient terhadap a:

dy/da = b

gradient terhadap b:

dy/db = a

Sedangkan matrix multiplication menggunakan aturan gradient matrix.

In [5]:
import numpy as np

A = np.array([
    [1, 2],
    [3, 4]
])

B = np.array([
    [5],
    [6]
])

C = A.dot(B)

print("A:")
print(A)

print("\nB:")
print(B)

print("\nA × B:")
print(C)

A:
[[1 2]
 [3 4]]

B:
[[5]
 [6]]

A × B:
[[17]
 [39]]


### Kenapa Matrix Multiplication Penting?

Operasi matrix multiplication merupakan dasar dari layer Linear/Dense.

Secara sederhana:

output = input × weight

Kemudian hasil tersebut dapat diteruskan ke activation function.

Jadi:

Input
↓
Matrix Multiplication
↓
Activation
↓
Output

## 7. Menggunakan Autograd untuk Melatih Neural Network

Setelah autograd dapat menangani berbagai operasi, sistem tersebut dapat digunakan untuk neural network.

Alur training:

1. Input masuk ke network.
2. Network menghasilkan prediction.
3. Prediction dibandingkan dengan target.
4. Loss dihitung.
5. `.backward()` menghitung gradient.
6. Optimizer memperbarui weight.
7. Proses diulang.

Dengan demikian kita kembali ke konsep:

**Predict → Compare → Learn**

Bedanya, proses gradient dan pembaruan parameter sekarang mulai diotomatisasi oleh framework.

## 8. Automatic Optimization

Setelah gradient diperoleh, parameter neural network harus diperbarui.

Konsep dasarnya:

weight_new = weight_old - learning_rate × gradient

Optimizer bertugas melakukan proses tersebut secara otomatis.

Pada framework sederhana di buku digunakan optimizer SGD (Stochastic Gradient Descent).

Langkahnya:

1. Hitung prediction.
2. Hitung loss.
3. Jalankan backward().
4. Ambil gradient setiap parameter.
5. Update parameter.
6. Ulangi.

In [6]:
# Contoh sederhana SGD

weight = 0.0
learning_rate = 0.1

# Target yang ingin dicapai
target = 10.0

for i in range(10):

    # Prediction
    prediction = weight

    # Error
    error = prediction - target

    # Gradient sederhana
    gradient = error

    # Update weight
    weight = weight - learning_rate * gradient

    print(f"Step {i+1}: weight = {weight:.4f}")

Step 1: weight = 1.0000
Step 2: weight = 1.9000
Step 3: weight = 2.7100
Step 4: weight = 3.4390
Step 5: weight = 4.0951
Step 6: weight = 4.6856
Step 7: weight = 5.2170
Step 8: weight = 5.6953
Step 9: weight = 6.1258
Step 10: weight = 6.5132


### Hasil

Perhatikan bahwa nilai weight secara bertahap mendekati target 10.

Ini menggambarkan bagaimana optimizer mengurangi error dengan menggunakan gradient.

Semakin baik proses optimasi, semakin kecil error yang diperoleh.

## 9. Layer

Framework menjadi lebih mudah digunakan jika neural network dibangun dari layer.

Contoh layer:

- Linear
- Tanh
- Sigmoid
- Embedding
- Cross Entropy
- RNN

Setiap layer memiliki tugas tertentu.

Contoh:

Linear → melakukan transformasi data
Tanh → memberikan nonlinearity
Embedding → mengubah index menjadi vector
CrossEntropy → menghitung loss klasifikasi
RNN → memproses sequence

In [7]:
# Contoh konsep arsitektur neural network

import numpy as np

np.random.seed(0)

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

W1 = np.random.randn(2, 3)
W2 = np.random.randn(3, 1)

# Layer 1
hidden = np.tanh(X.dot(W1))

# Layer 2
output = hidden.dot(W2)

print("Input:")
print(X)

print("\nHidden layer:")
print(hidden)

print("\nOutput:")
print(output)

Input:
[[0 0]
 [0 1]
 [1 0]
 [1 1]]

Hidden layer:
[[ 0.          0.          0.        ]
 [ 0.97762674  0.95337222 -0.7518851 ]
 [ 0.94295388  0.38008347  0.75251907]
 [ 0.9993359   0.97878291  0.0014601 ]]

Output:
[[0.        ]
 [0.8621408 ]
 [0.76068704]
 [0.8011609 ]]


## 10. Nonlinearity Layer

Neural network membutuhkan fungsi nonlinearity agar dapat mempelajari hubungan yang kompleks.

Dua fungsi yang digunakan dalam framework:

### Tanh

tanh(x)

Nilainya berada antara -1 dan 1.

### Sigmoid

sigmoid(x) = 1 / (1 + e^-x)

Nilainya berada antara 0 dan 1.

Keduanya juga memiliki gradient yang dapat dihitung oleh autograd.

In [8]:
import numpy as np

x = np.linspace(-5, 5, 11)

tanh = np.tanh(x)
sigmoid = 1 / (1 + np.exp(-x))

print("x:")
print(x)

print("\nTanh:")
print(tanh)

print("\nSigmoid:")
print(sigmoid)

x:
[-5. -4. -3. -2. -1.  0.  1.  2.  3.  4.  5.]

Tanh:
[-0.9999092  -0.9993293  -0.99505475 -0.96402758 -0.76159416  0.
  0.76159416  0.96402758  0.99505475  0.9993293   0.9999092 ]

Sigmoid:
[0.00669285 0.01798621 0.04742587 0.11920292 0.26894142 0.5
 0.73105858 0.88079708 0.95257413 0.98201379 0.99330715]


## 11. Embedding Layer

Embedding digunakan untuk mengubah sebuah index menjadi vector.

Misalnya kita memiliki vocabulary:

0 = "saya"
1 = "makan"
2 = "nasi"

Maka:

index 0 → [0.2, 0.7, 0.1]
index 1 → [0.8, 0.3, 0.5]
index 2 → [0.4, 0.9, 0.6]

Vector tersebut merupakan representasi numerik dari sebuah kata.

Embedding sangat penting dalam NLP karena neural network tidak dapat langsung memahami kata sebagai teks.

In [9]:
import numpy as np

vocab = {
    "saya": 0,
    "makan": 1,
    "nasi": 2
}

embedding = np.array([
    [0.2, 0.7, 0.1],
    [0.8, 0.3, 0.5],
    [0.4, 0.9, 0.6]
])

word = "makan"
index = vocab[word]

print("Kata:", word)
print("Index:", index)
print("Embedding:", embedding[index])

Kata: makan
Index: 1
Embedding: [0.8 0.3 0.5]


## 12. Cross-Entropy Loss

Cross entropy digunakan untuk mengukur seberapa jauh prediction model dari target sebenarnya pada masalah klasifikasi.

Secara sederhana:

- Prediction benar dengan confidence tinggi → loss kecil
- Prediction salah dengan confidence tinggi → loss besar

Dalam framework pada buku, softmax dan cross entropy digabungkan dalam satu loss function.

Gradient sederhananya menjadi:

gradient = softmax_output - target_distribution

Hal ini membuat proses training menjadi lebih efisien.

In [10]:
import numpy as np

# Prediction model setelah softmax
prediction = np.array([0.1, 0.7, 0.2])

# Target = class 1
target = 1

loss = -np.log(prediction[target])

print("Prediction:", prediction)
print("Target class:", target)
print("Cross entropy loss:", loss)

Prediction: [0.1 0.7 0.2]
Target class: 1
Cross entropy loss: 0.35667494393873245


### Interpretasi

Model memberikan probabilitas:

Class 0 = 10%
Class 1 = 70%
Class 2 = 20%

Target sebenarnya adalah Class 1.

Karena model memberikan probabilitas terbesar kepada class yang benar, loss relatif kecil.

Jika probabilitas class 1 hanya 1%, maka loss akan jauh lebih besar.

## 13. Recurrent Neural Network Layer

Setelah framework memiliki:

- Tensor
- Autograd
- Optimizer
- Linear layer
- Nonlinearity
- Embedding
- Cross Entropy

framework dapat digunakan untuk membangun RNN.

RNN digunakan untuk data sequence seperti:

- teks
- kalimat
- time series
- audio

RNN mempertahankan hidden state yang membawa informasi dari timestep sebelumnya.

Secara sederhana:

input_t + hidden_(t-1)
↓
RNN
↓
hidden_t
↓
output_t

Hidden state memungkinkan informasi dari sequence sebelumnya digunakan untuk membuat prediksi berikutnya.

In [11]:
import numpy as np

np.random.seed(0)

input_size = 3
hidden_size = 4

W = np.random.randn(input_size, hidden_size)
U = np.random.randn(hidden_size, hidden_size)

hidden = np.zeros(hidden_size)

sequence = [
    np.array([1, 0, 0]),
    np.array([0, 1, 0]),
    np.array([0, 0, 1])
]

for i, x in enumerate(sequence):

    hidden = np.tanh(
        x.dot(W) + hidden.dot(U)
    )

    print(f"Timestep {i+1}")
    print(hidden)

Timestep 1
[0.94295388 0.38008347 0.75251907 0.97762674]
Timestep 2
[ 0.9979899  -0.9536383   0.97490792 -0.71767531]
Timestep 3
[-0.99988605  0.9839528   0.79991174  0.964905  ]


### Yang Terjadi

Pada setiap timestep:

hidden baru = tanh(input × W + hidden lama × U)

Jadi hidden state tidak dibuat dari input saat ini saja.

Hidden state membawa sebagian informasi dari timestep sebelumnya.

Inilah konsep dasar RNN.

# Kesimpulan Chapter 13

Chapter ini menjelaskan bagaimana deep learning framework bekerja dari dalam.

Konsep utamanya:

1. **Tensor**
   Digunakan untuk menyimpan dan memproses data numerik.

2. **Autograd**
   Menghitung gradient secara otomatis melalui computational graph.

3. **Backpropagation**
   Menyebarkan gradient dari output kembali menuju parameter.

4. **Optimizer**
   Menggunakan gradient untuk memperbarui parameter model.

5. **Layer**
   Membungkus operasi tertentu agar neural network mudah dibangun.

6. **Embedding**
   Mengubah index menjadi representasi vector.

7. **Cross Entropy**
   Mengukur error pada klasifikasi.

8. **RNN**
   Memproses data sequence menggunakan hidden state.

Alur lengkap:

Input
↓
Tensor
↓
Layer
↓
Forward Propagation
↓
Loss
↓
Autograd
↓
Backpropagation
↓
Gradient
↓
Optimizer
↓
Update Parameter

Framework seperti PyTorch pada dasarnya mengotomatisasi banyak proses tersebut.

Jadi tujuan utama chapter ini bukan hanya bisa menggunakan framework, tetapi memahami apa yang sebenarnya dilakukan framework di balik layar.

Chapter berikutnya akan menggunakan RNN untuk masalah yang lebih kompleks dan memperkenalkan **Long Short-Term Memory (LSTM)**.